In [ ]:
# Cell: Kaggle protobuf compatibility, imports, and configuration
# If Internet is enabled in Kaggle, align protobuf runtime with TF metadata's generated protos.
import subprocess, sys, importlib.metadata
try:
    _pbv = importlib.metadata.version("protobuf")
    _major_minor = tuple(int(x) for x in _pbv.split(".")[:2])
    if _major_minor < (6, 31):
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "protobuf>=6.31.1,<7"])
except Exception as _e:
    print("Protobuf auto-fix did not complete. Enable Kaggle Internet and rerun if the same protobuf error appears:", _e)

import os, time, gc, random, re
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import tensorflow as tf
import tensorflow_datasets as tfds
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

# Configure publication figures: Times New Roman, EPS, and 600 dpi
# Use Times New Roman when installed; otherwise use an available serif fallback.
_available_fonts = {f.name for f in fm.fontManager.ttflist}
_font_choice = "Times New Roman" if "Times New Roman" in _available_fonts else (
    "Liberation Serif" if "Liberation Serif" in _available_fonts else "DejaVu Serif"
)
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = [_font_choice, "DejaVu Serif"]
plt.rcParams["savefig.dpi"] = 600
plt.rcParams["figure.dpi"] = 600
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["pdf.fonttype"] = 42

OUTPUT_ROOT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
PLOT_DIR = os.path.join(OUTPUT_ROOT, "cs3807_attention_outputs")
os.makedirs(PLOT_DIR, exist_ok=True)
_plot_counter = 0

def save_open_figures():
    """Save all currently open figures as 600 dpi EPS files."""
    global _plot_counter
    for _fig_num in plt.get_fignums():
        _fig = plt.figure(_fig_num)
        _plot_counter += 1
        _path = os.path.join(PLOT_DIR, f"figure_{_plot_counter:03d}.eps")
        _fig.savefig(_path, format="eps", dpi=600, bbox_inches="tight")
        print(f"Saved EPS: {_path}")

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
IMG_SIZE, NUM_CLASSES, BATCH_SIZE = 224, 5, 8
AUTOTUNE = 1
QUICK_RUN = False
EPOCHS = 5 if not QUICK_RUN else 2
FINE_TUNE_EPOCHS = 3 if not QUICK_RUN else 1
RUN_EXTENDED_ABLATIONS = True
RUN_MULTISTAGE_ABLATION = True
RUN_COMBINED_ATTENTION = True
for _gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(_gpu, True)
    except RuntimeError:
        pass
print("TensorFlow:", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
# Kaggle/Colab output paths and memory-safe settings
from pathlib import Path
IN_KAGGLE = Path("/kaggle/input").exists()
OUTPUT_ROOT = Path("/kaggle/working") if IN_KAGGLE else Path("/content")
OUTPUT_DIR = OUTPUT_ROOT / "cs3807_attention_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR = str(OUTPUT_DIR)
print("Runtime:", "Kaggle" if IN_KAGGLE else "Colab/other")
print("Output directory:", OUTPUT_DIR)


In [ ]:
# Check Kaggle's preinstalled packages without replacing its TensorFlow stack
import importlib.metadata as metadata
for package in ("tensorflow", "keras", "matplotlib", "scikit-learn", "pandas", "numpy"):
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: not installed")


In [ ]:
# Cell: Load TensorFlow Flowers labels and create a fixed stratified 70/15/15 split
(ds_all, ds_info) = tfds.load("tf_flowers", split="train", as_supervised=True, with_info=True, shuffle_files=False)
class_names = ds_info.features["label"].names
labels_np = np.asarray([int(label) for _, label in tfds.as_numpy(ds_all)], dtype=np.int64)
indices = np.arange(len(labels_np))
train_idx, temp_idx = train_test_split(indices, test_size=0.30, random_state=SEED, stratify=labels_np)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, random_state=SEED, stratify=labels_np[temp_idx])

def get_raw_image(index):
    image, _ = next(iter(ds_all.skip(int(index)).take(1)))
    return image.numpy()

print("Classes:", class_names)
print("Samples:", len(labels_np), "| Train/validation/test:", len(train_idx), len(val_idx), len(test_idx))


In [ ]:
# Cell: Show sample images and prepare memory-efficient tf.data datasets
fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(15, 3))
for c, ax in enumerate(axes):
    i = int(next(i for i in train_idx if labels_np[i] == c))
    ax.imshow(get_raw_image(i)); ax.set_title(class_names[c]); ax.axis("off")
plt.tight_layout(); save_open_figures(); plt.show()

def preprocess(image, label, training=False):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32)
    if training:
        image = tf.image.random_flip_left_right(image)
        image = tf.image.random_brightness(image, max_delta=0.08)
        image = tf.image.random_contrast(image, 0.9, 1.1)
    image = tf.keras.applications.resnet50.preprocess_input(image)
    return image, tf.cast(label, tf.int32)

def make_dataset(indices, training=False):
    selected = set(int(i) for i in indices)
    def generator():
        for i, (image, label) in enumerate(tfds.as_numpy(ds_all)):
            if i in selected:
                yield image, np.int64(label)
    ds = tf.data.Dataset.from_generator(
        generator,
        output_signature=(tf.TensorSpec((None, None, 3), tf.uint8), tf.TensorSpec((), tf.int64))
    )
    if training:
        ds = ds.shuffle(len(indices), seed=SEED, reshuffle_each_iteration=True)
    return ds.map(lambda x, y: preprocess(x, y, training), num_parallel_calls=1).batch(BATCH_SIZE).prefetch(1)

train_ds = make_dataset(train_idx, True)
val_ds = make_dataset(val_idx)
test_ds = make_dataset(test_idx)


In [ ]:
# Cell: Channel attention implementation, evaluation, and plot
class ChannelAttention(tf.keras.layers.Layer):
    def __init__(self, reduction_ratio=16, **kwargs):
        super().__init__(**kwargs); self.reduction_ratio = reduction_ratio
    def build(self, input_shape):
        c = int(input_shape[-1]); r = max(1, c // self.reduction_ratio)
        self.fc1 = tf.keras.layers.Dense(r, activation="relu")
        self.fc2 = tf.keras.layers.Dense(c)
        super().build(input_shape)
    def call(self, inputs, return_attention=False):
        z = tf.reduce_mean(inputs, axis=[1, 2])
        weights = tf.sigmoid(self.fc2(self.fc1(z)))[:, None, None, :]
        out = inputs * weights
        return (out, weights) if return_attention else out

dummy = tf.random.normal((2, 14, 14, 1024), seed=SEED)
ca = ChannelAttention()
ca_out, ca_weights = ca(dummy, return_attention=True)
print("Input:", dummy.shape, "| Weights:", ca_weights.shape, "| Output:", ca_out.shape)
print("Parameters:", ca.count_params(), "| Weight range:", float(tf.reduce_min(ca_weights)), float(tf.reduce_max(ca_weights)))
assert tuple(ca_out.shape) == tuple(dummy.shape) and tuple(ca_weights.shape) == (2, 1, 1, 1024)
# Trained channel weights are visualized after model training.

In [ ]:
# Cell: Spatial attention implementation, evaluation, and plot
class SpatialAttention(tf.keras.layers.Layer):
    def __init__(self, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        if kernel_size not in (3, 5, 7): raise ValueError("kernel_size must be 3, 5, or 7")
        self.kernel_size = kernel_size
        self.conv = tf.keras.layers.Conv2D(1, kernel_size, padding="same", use_bias=False)
    def call(self, inputs, return_attention=False):
        avg = tf.reduce_mean(inputs, axis=-1, keepdims=True)
        mx = tf.reduce_max(inputs, axis=-1, keepdims=True)
        attention = tf.sigmoid(self.conv(tf.concat([avg, mx], axis=-1)))
        out = inputs * attention
        return (out, attention) if return_attention else out

sa = SpatialAttention(7)
sa_out, sa_map = sa(dummy, return_attention=True)
print("Input:", dummy.shape, "| Map:", sa_map.shape, "| Output:", sa_out.shape)
print("Parameters:", sa.count_params(), "| Weight range:", float(tf.reduce_min(sa_map)), float(tf.reduce_max(sa_map)))
assert tuple(sa_out.shape) == tuple(dummy.shape) and tuple(sa_map.shape) == (2, 14, 14, 1)
assert sa.count_params() == 98
plt.figure(figsize=(5, 4)); plt.imshow(sa_map[0, :, :, 0].numpy(), cmap="viridis", vmin=0, vmax=1)
plt.colorbar(label="Weight"); plt.title("Spatial Attention Map (untrained module)")
plt.tight_layout(); save_open_figures(); plt.show()

In [ ]:
# Cell: Define sequential attention, self-attention, SE, and ECA
class ChannelSpatialAttention(tf.keras.layers.Layer):
    def __init__(self, reduction_ratio=16, kernel_size=7, **kwargs):
        super().__init__(**kwargs); self.channel = ChannelAttention(reduction_ratio, name=f"{self.name}_channel"); self.spatial = SpatialAttention(kernel_size, name=f"{self.name}_spatial")
    def call(self, inputs, return_attention=False):
        x = self.channel(inputs); x, amap = self.spatial(x, return_attention=True)
        return (x, amap) if return_attention else x

class SpatialChannelAttention(tf.keras.layers.Layer):
    def __init__(self, reduction_ratio=16, kernel_size=7, **kwargs):
        super().__init__(**kwargs); self.spatial = SpatialAttention(kernel_size, name=f"{self.name}_spatial"); self.channel = ChannelAttention(reduction_ratio, name=f"{self.name}_channel")
    def call(self, inputs):
        return self.channel(self.spatial(inputs))

class LightweightSelfAttention(tf.keras.layers.Layer):
    def __init__(self, key_dim=None, **kwargs):
        super().__init__(**kwargs); self.key_dim = key_dim
    def build(self, shape):
        c = int(shape[-1]); d = self.key_dim or max(8, c // 8)
        self.q = tf.keras.layers.Conv2D(d, 1, use_bias=False)
        self.k = tf.keras.layers.Conv2D(d, 1, use_bias=False)
        self.v = tf.keras.layers.Conv2D(c, 1, use_bias=False)
        self.gamma = self.add_weight(name="gamma", shape=(), initializer="zeros", trainable=True)
        super().build(shape)
    def call(self, x, return_attention=False):
        b, h, w = tf.shape(x)[0], tf.shape(x)[1], tf.shape(x)[2]
        q = tf.reshape(self.q(x), (b, h*w, -1)); k = tf.reshape(self.k(x), (b, h*w, -1))
        v = tf.reshape(self.v(x), (b, h*w, -1))
        a = tf.nn.softmax(tf.matmul(q, k, transpose_b=True) / tf.sqrt(tf.cast(tf.shape(q)[-1], tf.float32)), axis=-1)
        y = tf.reshape(tf.matmul(a, v), (b, h, w, int(x.shape[-1])))
        out = x + self.gamma * y
        return (out, a) if return_attention else out

class SEAttention(tf.keras.layers.Layer):
    def build(self, shape):
        c = int(shape[-1]); r = max(1, c // 16)
        self.fc1 = tf.keras.layers.Dense(r, activation="relu"); self.fc2 = tf.keras.layers.Dense(c, activation="sigmoid")
        super().build(shape)
    def call(self, x):
        z = tf.reduce_mean(x, axis=[1, 2]); return x * self.fc2(self.fc1(z))[:, None, None, :]

class ECAAttention(tf.keras.layers.Layer):
    def __init__(self, kernel_size=3, **kwargs):
        super().__init__(**kwargs); self.conv = tf.keras.layers.Conv1D(1, kernel_size, padding="same", use_bias=False)
    def call(self, x):
        z = tf.reduce_mean(x, axis=[1, 2])[:, :, None]
        weights = tf.transpose(tf.sigmoid(self.conv(z)), [0, 2, 1])[:, None, :, :]
        return x * weights

In [ ]:
# Cell: Build stage submodels and insert attention without breaking residual connections
def build_resnet_stage_models():
    base = tf.keras.applications.ResNet50(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
    stem = tf.keras.Model(base.input, base.get_layer("pool1_pool").output, name="resnet_stem")
    specs = [
        ("P1", "pool1_pool", "conv2_block3_out"),
        ("P2", "conv2_block3_out", "conv3_block4_out"),
        ("P3", "conv3_block4_out", "conv4_block6_out"),
        ("P4", "conv4_block6_out", "conv5_block3_out"),
    ]
    stages = {}
    for pos, in_name, out_name in specs:
        stages[pos] = tf.keras.Model(base.get_layer(in_name).output, base.get_layer(out_name).output, name=f"resnet_stage_{pos}")
    return base, stem, stages

def build_attention_classifier(attention_type="none", position="P3", order="channel_spatial", kernel_size=7, freeze_backbone=True, positions=None):
    if positions is None: positions = [position]
    if any(p not in ("P1", "P2", "P3", "P4") for p in positions): raise ValueError("positions must be P1, P2, P3, or P4")
    base, stem, stages = build_resnet_stage_models()
    inputs = tf.keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
    x = stem(inputs); amap = None; att_feature = None; channel_weights = None; self_weights = None
    for pos in ("P1", "P2", "P3", "P4"):
        x = stages[pos](x)
        if pos not in positions or attention_type == "none": continue
        if attention_type == "channel":
            layer = ChannelAttention(name=f"channel_attention_{pos.lower()}"); x, channel_weights = layer(x, return_attention=True); amap = channel_weights
        elif attention_type == "spatial":
            layer = SpatialAttention(kernel_size, name=f"spatial_attention_{pos.lower()}"); x, amap = layer(x, return_attention=True)
        elif attention_type in ("channel_spatial", "cbam"):
            if order == "channel_spatial":
                layer = ChannelSpatialAttention(kernel_size=kernel_size, name=f"channel_spatial_attention_{pos.lower()}")
                x, amap = layer(x, return_attention=True)
                channel_weights = layer.channel(x) if False else None
            else:
                layer = SpatialChannelAttention(kernel_size=kernel_size, name=f"spatial_channel_attention_{pos.lower()}")
                x = layer(x)
        elif attention_type == "combined":
            cs = ChannelSpatialAttention(kernel_size=kernel_size, name=f"combined_channel_spatial_{pos.lower()}")
            x, amap = cs(x, return_attention=True)
            layer = LightweightSelfAttention(name=f"combined_self_attention_{pos.lower()}")
            x, self_weights = layer(x, return_attention=True); amap = self_weights
        elif attention_type == "self":
            layer = LightweightSelfAttention(name=f"self_attention_{pos.lower()}"); x, self_weights = layer(x, return_attention=True); amap = self_weights
        elif attention_type == "se":
            layer = SEAttention(name=f"se_attention_{pos.lower()}"); x = layer(x)
        elif attention_type == "eca":
            layer = ECAAttention(name=f"eca_attention_{pos.lower()}"); x = layer(x)
        else: raise ValueError(f"Unknown attention_type: {attention_type}")
        att_feature = x

    final_features = x
    pooled = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
    pooled = tf.keras.layers.Dropout(0.3, name="classifier_dropout")(pooled)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", name="classifier")(pooled)
    model = tf.keras.Model(inputs, outputs, name=f"ResNet50_{attention_type}_{'_'.join(positions)}")

    if freeze_backbone:
        for layer in base.layers: layer.trainable = False
        for layer in model.layers:
            if any(layer.name.startswith(prefix) for prefix in (
                "channel_attention_", "spatial_attention_", "channel_spatial_attention_",
                "spatial_channel_attention_", "self_attention_", "combined_channel_spatial_",
                "combined_self_attention_", "se_attention_", "eca_attention_"
            )):
                layer.trainable = True
        model.get_layer("classifier").trainable = True

    model.gradcam_probe = tf.keras.Model(inputs, [final_features, outputs], name="gradcam_probe")
    model.attention_probe = tf.keras.Model(inputs, [amap, att_feature], name="attention_probe") if amap is not None else None
    model.channel_probe = tf.keras.Model(inputs, [channel_weights, att_feature], name="channel_probe") if channel_weights is not None else None
    model.self_attention_probe = tf.keras.Model(inputs, [self_weights, att_feature], name="self_attention_probe") if self_weights is not None else None
    return model, base

In [ ]:
# Cell: Verify ResNet50 stage shapes and classifier output before any training
base, stem, stages = build_resnet_stage_models()
z = stem(tf.zeros((1, IMG_SIZE, IMG_SIZE, 3)))
expected = {"P1": (56, 56, 256), "P2": (28, 28, 512), "P3": (14, 14, 1024), "P4": (7, 7, 2048)}
for pos in ("P1", "P2", "P3", "P4"):
    z = stages[pos](z)
    print(pos, tuple(z.shape[1:]))
    assert tuple(z.shape[1:]) == expected[pos]
del base, stem, stages, z
tf.keras.backend.clear_session(); gc.collect()

for pos in ("P1", "P2", "P3", "P4"):
    model, _ = build_attention_classifier("channel_spatial", pos)
    out = model(tf.zeros((2, IMG_SIZE, IMG_SIZE, 3)), training=False)
    print(pos, "output:", out.shape, "| parameters:", model.count_params())
    assert tuple(out.shape) == (2, NUM_CLASSES)
    del model
    tf.keras.backend.clear_session(); gc.collect()

In [ ]:
# Cell: Define training, metrics, inference-time, and learning-curve utilities
MODEL_CONFIGS = {}
CHECKPOINT_DIR = os.path.join(PLOT_DIR, "model_checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def checkpoint_model(label, model, attention_type, position, order="channel_spatial", kernel_size=7, positions=None):
    safe_label = re.sub(r"[^A-Za-z0-9_.-]+", "_", label)
    path = os.path.join(CHECKPOINT_DIR, safe_label + ".weights.h5")
    model.save_weights(path)
    MODEL_CONFIGS[label] = {
        "attention_type": attention_type, "position": position,
        "order": order, "kernel_size": kernel_size, "positions": positions
    }
    return path

def load_checkpoint_model(label):
    cfg = MODEL_CONFIGS[label]
    model, _base = build_attention_classifier(
        cfg["attention_type"], cfg["position"], order=cfg["order"],
        kernel_size=cfg["kernel_size"], positions=cfg["positions"]
    )
    model.load_weights(os.path.join(CHECKPOINT_DIR, re.sub(r"[^A-Za-z0-9_.-]+", "_", label) + ".weights.h5"))
    return model

def compile_model(model, lr=1e-3):
    model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

def evaluate_model(model, dataset):
    yt, yp = [], []
    first_x, _ = next(iter(dataset)); _ = model(first_x, training=False).numpy()
    start = time.perf_counter(); n = 0
    for x, y in dataset:
        p = model(x, training=False).numpy()
        yp.extend(np.argmax(p, axis=1)); yt.extend(y.numpy()); n += int(x.shape[0])
    elapsed = time.perf_counter() - start
    return {
        "accuracy": accuracy_score(yt, yp),
        "macro_precision": precision_score(yt, yp, average="macro", zero_division=0),
        "macro_recall": recall_score(yt, yp, average="macro", zero_division=0),
        "macro_f1": f1_score(yt, yp, average="macro", zero_division=0),
        "total_parameters": model.count_params(),
        "trainable_parameters": int(sum(np.prod(v.shape) for v in model.trainable_weights)),
        "inference_ms_per_image": 1000 * elapsed / max(n, 1),
        "y_true": np.asarray(yt), "y_pred": np.asarray(yp)
    }

def plot_history(history, title):
    h = history.history
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(h.get("accuracy", []), label="Train"); ax[0].plot(h.get("val_accuracy", []), label="Validation")
    ax[0].set_title(title + " accuracy"); ax[0].set_xlabel("Epoch"); ax[0].legend()
    ax[1].plot(h.get("loss", []), label="Train"); ax[1].plot(h.get("val_loss", []), label="Validation")
    ax[1].set_title(title + " loss"); ax[1].set_xlabel("Epoch"); ax[1].legend()
    plt.tight_layout(); save_open_figures(); plt.show()

def fine_tune_model(model, base, train_ds, val_ds, epochs=FINE_TUNE_EPOCHS):
    # Unfreeze only the final ResNet50 stage while preserving the same protocol across comparisons.
    for layer in base.layers:
        layer.trainable = layer.name.startswith("conv5_") or layer.name.startswith("pool5")
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    history = model.fit(train_ds, validation_data=val_ds, epochs=epochs, verbose=1)
    return history


In [ ]:
# Sub-experiment 1: Attention-Type Ablation at P3
type_specs = [
    ("Baseline", "none", "P3"),
    ("Channel_P3", "channel", "P3"),
    ("Spatial_P3", "spatial", "P3"),
    ("ChannelSpatial_P3", "channel_spatial", "P3"),
    ("SelfAttention_P3", "self", "P3"),
    ("CombinedAttention_P3", "combined", "P3"),
]
models, histories, results = {}, {}, []
for label, attn, pos in type_specs:
    print("\nTraining", label)
    tf.keras.backend.clear_session(); gc.collect()
    model, base = build_attention_classifier(attn, pos)
    compile_model(model)
    hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=1)
    fine_hist = fine_tune_model(model, base, train_ds, val_ds)
    plot_history(hist, label + " frozen-backbone")
    plot_history(fine_hist, label + " fine-tuning")
    metrics = evaluate_model(model, test_ds)
    metrics.update({"model": label, "attention_type": attn, "position": pos})
    print({k: v for k, v in metrics.items() if k not in ("y_true", "y_pred")})
    results.append(metrics); checkpoint_model(label, model, attn, pos); histories[label] = hist
    del model, base; tf.keras.backend.clear_session(); gc.collect()

results_df = pd.DataFrame([{k:v for k,v in r.items() if k not in ("y_true","y_pred")} for r in results])
display(results_df)


In [ ]:
# Sub-experiment 2: Attention-Position Ablation at P2, P3, and P4
position_results = []
for pos in ("P2", "P4"):
    label = f"ChannelSpatial_{pos}"
    print("\nTraining", label)
    tf.keras.backend.clear_session(); gc.collect()
    model, base = build_attention_classifier("channel_spatial", pos)
    compile_model(model)
    hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=1)
    fine_hist = fine_tune_model(model, base, train_ds, val_ds)
    plot_history(hist, label + " frozen-backbone")
    plot_history(fine_hist, label + " fine-tuning")
    m = evaluate_model(model, test_ds); m.update({"model": label, "attention_type": "channel_spatial", "position": pos})
    position_results.append(m); checkpoint_model(label, model, "channel_spatial", pos, positions=[pos]); histories[label] = hist
    del model, base; tf.keras.backend.clear_session(); gc.collect()

# Sub-experiment 3: Multiple Attention Positions at P3, P4, and P3 + P4
if RUN_MULTISTAGE_ABLATION:
    label = "ChannelSpatial_P3_P4"
    print("\nTraining", label)
    tf.keras.backend.clear_session(); gc.collect()
    model, base = build_attention_classifier("channel_spatial", "P3", positions=["P3", "P4"])
    compile_model(model)
    hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=1)
    fine_hist = fine_tune_model(model, base, train_ds, val_ds)
    plot_history(hist, label + " frozen-backbone"); plot_history(fine_hist, label + " fine-tuning")
    m = evaluate_model(model, test_ds); m.update({"model": label, "attention_type": "channel_spatial", "position": "P3+P4"})
    position_results.append(m); checkpoint_model(label, model, "channel_spatial", "P3", positions=["P3", "P4"]); histories[label] = hist
    del model, base; tf.keras.backend.clear_session(); gc.collect()
position_df = pd.DataFrame([{k:v for k,v in r.items() if k not in ("y_true","y_pred")} for r in position_results])
display(position_df)


In [ ]:
# Sub-experiment 4: Attention Order (Channel → Spatial vs Spatial → Channel)
extended_results = []
if RUN_EXTENDED_ABLATIONS:
    order_specs = [
        ("SpatialToChannel_P3", "channel_spatial", "P3", "spatial_channel", 7),
    ]
    for label, attn, pos, order, kernel in order_specs:
        tf.keras.backend.clear_session(); gc.collect()
        model, base = build_attention_classifier(attn, pos, order=order, kernel_size=kernel)
        compile_model(model)
        hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=1)
        fine_hist = fine_tune_model(model, base, train_ds, val_ds)
        plot_history(hist, label + " frozen-backbone"); plot_history(fine_hist, label + " fine-tuning")
        m = evaluate_model(model, test_ds); m.update({"model": label, "attention_type": attn, "position": pos})
        extended_results.append(m); checkpoint_model(label, model, attn, pos, order=order, kernel_size=kernel)
        del model, base; tf.keras.backend.clear_session(); gc.collect()

    # Sub-experiment 5: Spatial Attention Kernel Size (3×3, 5×5, and 7×7)
    kernel_specs = [
        ("Spatial_k3_P3", "spatial", "P3", "channel_spatial", 3),
        ("Spatial_k5_P3", "spatial", "P3", "channel_spatial", 5),
        ("Spatial_k7_P3", "spatial", "P3", "channel_spatial", 7),
    ]
    for label, attn, pos, order, kernel in kernel_specs:
        tf.keras.backend.clear_session(); gc.collect()
        model, base = build_attention_classifier(attn, pos, order=order, kernel_size=kernel)
        compile_model(model)
        hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=1)
        fine_hist = fine_tune_model(model, base, train_ds, val_ds)
        plot_history(hist, label + " frozen-backbone"); plot_history(fine_hist, label + " fine-tuning")
        m = evaluate_model(model, test_ds); m.update({"model": label, "attention_type": attn, "position": pos})
        extended_results.append(m); checkpoint_model(label, model, attn, pos, order=order, kernel_size=kernel)
        del model, base; tf.keras.backend.clear_session(); gc.collect()

    # Sub-experiment 6: Additional Attention Mechanisms (SE, ECA, and CBAM)
    additional_specs = [
        ("SE_P3", "se", "P3", "channel_spatial", 7),
        ("ECA_P3", "eca", "P3", "channel_spatial", 7),
        ("CBAM_P3", "cbam", "P3", "channel_spatial", 7),
    ]
    for label, attn, pos, order, kernel in additional_specs:
        tf.keras.backend.clear_session(); gc.collect()
        model, base = build_attention_classifier(attn, pos, order=order, kernel_size=kernel)
        compile_model(model)
        hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=1)
        fine_hist = fine_tune_model(model, base, train_ds, val_ds)
        plot_history(hist, label + " frozen-backbone"); plot_history(fine_hist, label + " fine-tuning")
        m = evaluate_model(model, test_ds); m.update({"model": label, "attention_type": attn, "position": pos})
        extended_results.append(m); checkpoint_model(label, model, attn, pos, order=order, kernel_size=kernel)
        del model, base; tf.keras.backend.clear_session(); gc.collect()
extended_df = pd.DataFrame([{k:v for k,v in r.items() if k not in ("y_true","y_pred")} for r in extended_results])
display(extended_df)


In [ ]:
# Cell: Generate required performance graphs and display all experiment result tables
all_rows = results + position_results + extended_results
all_metrics = pd.DataFrame([{k:v for k,v in r.items() if k not in ("y_true", "y_pred")} for r in all_rows])
all_metrics = all_metrics.drop_duplicates(subset=["model"], keep="last").reset_index(drop=True)

# Required accuracy and macro-F1 bar charts
for metric, label in (("accuracy", "Test accuracy"), ("macro_f1", "Macro F1-score")):
    plt.figure(figsize=(max(12, 0.65 * len(all_metrics)), 5))
    plt.bar(all_metrics["model"], all_metrics[metric])
    plt.ylabel(label); plt.title(f"Attention mechanism versus {label}")
    plt.xticks(rotation=65, ha="right"); plt.tight_layout(); save_open_figures(); plt.show()

# Required accuracy–complexity and macro-F1–inference-time plots
plt.figure(figsize=(9, 6)); plt.scatter(all_metrics["total_parameters"], all_metrics["accuracy"])
for _, row in all_metrics.iterrows(): plt.annotate(row["model"], (row["total_parameters"], row["accuracy"]), fontsize=7)
plt.xlabel("Total model parameters"); plt.ylabel("Test accuracy"); plt.title("Accuracy–complexity trade-off")
plt.tight_layout(); save_open_figures(); plt.show()
plt.figure(figsize=(9, 6)); plt.scatter(all_metrics["inference_ms_per_image"], all_metrics["macro_f1"])
for _, row in all_metrics.iterrows(): plt.annotate(row["model"], (row["inference_ms_per_image"], row["macro_f1"]), fontsize=7)
plt.xlabel("Inference time (ms/image)"); plt.ylabel("Macro F1-score"); plt.title("Macro F1–inference-time trade-off")
plt.tight_layout(); save_open_figures(); plt.show()

# Required main result table (all values are measured on the held-out test split)
main_names = ["Baseline", "Channel_P3", "Spatial_P3", "ChannelSpatial_P2", "ChannelSpatial_P3", "ChannelSpatial_P4", "SelfAttention_P3", "CombinedAttention_P3", "SE_P3", "ECA_P3", "CBAM_P3"]
main_table = all_metrics[all_metrics["model"].isin(main_names)].copy()
main_table = main_table[["model", "position", "accuracy", "macro_precision", "macro_recall", "macro_f1", "trainable_parameters", "total_parameters", "inference_ms_per_image"]]
print("MAIN RESULT TABLE")
display(main_table)

# Required attention-position result table
position_names = ["ChannelSpatial_P2", "ChannelSpatial_P3", "ChannelSpatial_P4", "ChannelSpatial_P3_P4"]
position_table = all_metrics[all_metrics["model"].isin(position_names)][["model", "position", "accuracy", "macro_precision", "macro_recall", "macro_f1", "trainable_parameters", "total_parameters", "inference_ms_per_image"]]
print("ATTENTION POSITION AND MULTI-STAGE TABLE")
display(position_table)

# Required attention-order table
order_names = ["ChannelSpatial_P3", "SpatialToChannel_P3"]
order_table = all_metrics[all_metrics["model"].isin(order_names)][["model", "accuracy", "macro_precision", "macro_recall", "macro_f1", "trainable_parameters", "total_parameters", "inference_ms_per_image"]]
print("ATTENTION ORDER TABLE")
display(order_table)

# Required spatial-kernel-size table
kernel_names = ["Spatial_k3_P3", "Spatial_k5_P3", "Spatial_k7_P3"]
kernel_table = all_metrics[all_metrics["model"].isin(kernel_names)].copy()
if not kernel_table.empty:
    kernel_table["spatial_kernel"] = kernel_table["model"].map({"Spatial_k3_P3": "3x3", "Spatial_k5_P3": "5x5", "Spatial_k7_P3": "7x7"})
    kernel_table = kernel_table[["spatial_kernel", "accuracy", "macro_f1", "inference_ms_per_image", "trainable_parameters", "total_parameters"]]
print("SPATIAL KERNEL SIZE TABLE")
display(kernel_table)

# Required additional attention-mechanism comparison table
additional_names = ["Baseline", "Channel_P3", "Spatial_P3", "SE_P3", "ECA_P3", "CBAM_P3", "SelfAttention_P3"]
additional_table = all_metrics[all_metrics["model"].isin(additional_names)][["model", "accuracy", "macro_precision", "macro_recall", "macro_f1", "trainable_parameters", "total_parameters", "inference_ms_per_image"]]
print("ADDITIONAL ATTENTION MECHANISM TABLE")
display(additional_table)

# Export filled tables after successful evaluations
main_table.to_csv(os.path.join(PLOT_DIR, "main_result_table.csv"), index=False)
position_table.to_csv(os.path.join(PLOT_DIR, "position_multistage_table.csv"), index=False)
order_table.to_csv(os.path.join(PLOT_DIR, "attention_order_table.csv"), index=False)
kernel_table.to_csv(os.path.join(PLOT_DIR, "spatial_kernel_table.csv"), index=False)
additional_table.to_csv(os.path.join(PLOT_DIR, "additional_attention_table.csv"), index=False)
all_metrics.to_csv(os.path.join(PLOT_DIR, "attention_experiment_results.csv"), index=False)


In [ ]:
# Cell: Confusion matrices and classification reports for baseline and attention model
for selected_name in ("Baseline", "ChannelSpatial_P3"):
    selected_metrics = next(r for r in results + position_results + extended_results if r["model"] == selected_name)
    cm = confusion_matrix(selected_metrics["y_true"], selected_metrics["y_pred"], labels=np.arange(NUM_CLASSES))
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(cmap="Blues", xticks_rotation=45)
    plt.title(f"Confusion matrix: {selected_name}"); plt.tight_layout(); save_open_figures(); plt.show()
    print(selected_name)
    print(classification_report(selected_metrics["y_true"], selected_metrics["y_pred"], labels=np.arange(NUM_CLASSES), target_names=class_names, zero_division=0))

In [ ]:
# Cell: Load selected checkpoints and visualize attention maps, features, and overlays
models = {}
for _name in ("Baseline", "Channel_P3", "Spatial_P3", "ChannelSpatial_P3", "SelfAttention_P3",
              "ChannelSpatial_P2", "ChannelSpatial_P4"):
    if _name in MODEL_CONFIGS:
        models[_name] = load_checkpoint_model(_name)

sample_id = int(test_idx[0])
raw = get_raw_image(sample_id)
img = tf.keras.applications.resnet50.preprocess_input(
    tf.cast(tf.image.resize(raw, (IMG_SIZE, IMG_SIZE)), tf.float32)
)[None, ...]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(raw); axes[0].set_title("Input"); axes[0].axis("off")
for ax, pos in zip(axes[1:], ("P2", "P3", "P4")):
    model_name = f"ChannelSpatial_{pos}"
    if model_name in models and models[model_name].attention_probe is not None:
        attention_map, refined_features = models[model_name].attention_probe(img, training=False)
        attention_map = tf.image.resize(attention_map[0], (IMG_SIZE, IMG_SIZE)).numpy().squeeze()
        ax.imshow(attention_map, cmap="viridis", vmin=0, vmax=1); ax.set_title(f"Spatial attention {pos}")
    else: ax.set_title(f"{pos} unavailable")
    ax.axis("off")
plt.tight_layout(); save_open_figures(); plt.show()

probe_model = models.get("ChannelSpatial_P3", models.get("Spatial_P3"))
if probe_model is not None and probe_model.attention_probe is not None:
    spatial_map, refined_features = probe_model.attention_probe(img, training=False)
    heat = tf.image.resize(spatial_map[0], (IMG_SIZE, IMG_SIZE)).numpy().squeeze()
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(raw); axes[0].set_title("Input")
    axes[1].imshow(heat, cmap="viridis", vmin=0, vmax=1); axes[1].set_title("Spatial attention map")
    axes[2].imshow(tf.image.resize(raw, (IMG_SIZE, IMG_SIZE)).numpy().astype(np.uint8)); axes[2].imshow(heat, cmap="jet", alpha=0.4); axes[2].set_title("Attention overlay")
    for ax in axes: ax.axis("off")
    plt.tight_layout(); save_open_figures(); plt.show()

if "Channel_P3" in models and models["Channel_P3"].channel_probe is not None:
    channel_weights, _ = models["Channel_P3"].channel_probe(img, training=False)
    w = channel_weights[0, 0, 0].numpy(); ids = np.argsort(w)[-20:][::-1]
    plt.figure(figsize=(10, 4)); plt.bar(range(len(ids)), w[ids]); plt.xticks(range(len(ids)), ids, rotation=45)
    plt.xlabel("Channel index"); plt.ylabel("Learned weight"); plt.title("Top 20 trained channel-attention weights")
    plt.tight_layout(); save_open_figures(); plt.show()

if "SelfAttention_P3" in models and models["SelfAttention_P3"].self_attention_probe is not None:
    attention_matrix, _ = models["SelfAttention_P3"].self_attention_probe(img, training=False)
    attention_matrix = attention_matrix[0].numpy(); n = attention_matrix.shape[0]
    query_idx = n // 2
    response = attention_matrix[query_idx].reshape(int(np.sqrt(n)), int(np.sqrt(n)))
    response = tf.image.resize(response[..., None], (IMG_SIZE, IMG_SIZE)).numpy().squeeze()
    fig, axes = plt.subplots(1, 2, figsize=(8, 4)); axes[0].imshow(raw); axes[0].set_title("Input")
    axes[1].imshow(response, cmap="viridis"); axes[1].set_title("Self-attention response")
    for ax in axes: ax.axis("off")
    plt.tight_layout(); save_open_figures(); plt.show()

# Feature maps before and after attention
if probe_model is not None:
    before = probe_model.gradcam_probe(img, training=False)[0]
    plt.figure(figsize=(9, 4)); plt.subplot(1, 2, 1)
    plt.imshow(tf.reduce_mean(before[0], axis=-1).numpy(), cmap="viridis"); plt.title("Final feature map"); plt.axis("off")
    plt.subplot(1, 2, 2)
    plt.imshow(tf.reduce_mean(refined_features[0], axis=-1).numpy(), cmap="viridis"); plt.title("Feature map after attention"); plt.axis("off")
    plt.tight_layout(); save_open_figures(); plt.show()


In [ ]:
# Cell: Grad-CAM comparison for baseline and attention models
def gradcam(model, batch, class_id=None):
    with tf.GradientTape() as tape:
        features, probs = model.gradcam_probe(batch, training=False)
        if class_id is None: class_id = tf.argmax(probs[0])
        score = probs[:, class_id]
    grads = tape.gradient(score, features)
    weights = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)
    heat = tf.maximum(tf.reduce_sum(features * weights, axis=-1), 0)
    heat /= tf.reduce_max(heat, axis=(1, 2), keepdims=True) + 1e-8
    return heat[0].numpy(), int(class_id.numpy()), probs[0].numpy()

def show_gradcam_comparison(raw_image, names):
    resized = tf.image.resize(raw_image, (IMG_SIZE, IMG_SIZE))
    batch = tf.keras.applications.resnet50.preprocess_input(tf.cast(resized, tf.float32))[None, ...]
    fig, axes = plt.subplots(1, len(names)+1, figsize=(4*(len(names)+1), 4))
    axes[0].imshow(raw_image); axes[0].set_title("Input"); axes[0].axis("off")
    for ax, name in zip(axes[1:], names):
        heat, cls, probs = gradcam(models[name], batch)
        heat = tf.image.resize(heat[..., None], (IMG_SIZE, IMG_SIZE)).numpy().squeeze()
        ax.imshow(raw_image); ax.imshow(heat, cmap="jet", alpha=0.4, vmin=0, vmax=1)
        ax.set_title(f"{name}\nPred: {class_names[cls]}"); ax.axis("off")
    plt.tight_layout(); save_open_figures(); plt.show()

gradcam_names = [n for n in ("Baseline", "Channel_P3", "ChannelSpatial_P3") if n in models]
show_gradcam_comparison(get_raw_image(test_idx[0]), gradcam_names)

In [ ]:
# Cell: Compare multiple models on three correct and two incorrect test images
def predict_all(model, dataset):
    yt, yp = [], []
    for x, y in dataset:
        yp.extend(np.argmax(model(x, training=False).numpy(), axis=1)); yt.extend(y.numpy())
    return np.asarray(yt), np.asarray(yp)

reference_name = "ChannelSpatial_P3" if "ChannelSpatial_P3" in models else "Baseline"
yt, yp = predict_all(models[reference_name], test_ds)
correct = np.where(yt == yp)[0][:3]; wrong = np.where(yt != yp)[0][:2]
compare_names = [n for n in ("Baseline", "Channel_P3", "Spatial_P3", "ChannelSpatial_P3", "SelfAttention_P3") if n in models]
for p in list(correct) + list(wrong):
    idx = int(test_idx[p]); raw = get_raw_image(idx)
    show_gradcam_comparison(raw, compare_names)
    plt.figure(figsize=(4, 4)); plt.imshow(raw)
    plt.title(f"True: {class_names[yt[p]]} | {reference_name}: {class_names[yp[p]]} | {'Correct' if yt[p] == yp[p] else 'Incorrect'}")
    plt.axis("off"); save_open_figures(); plt.show()

In [ ]:
# Cell: Export every filled table, training histories, figures, and discussion checklist
all_metrics.to_csv(os.path.join(PLOT_DIR, "attention_experiment_results.csv"), index=False)
main_table.to_csv(os.path.join(PLOT_DIR, "main_result_table.csv"), index=False)
position_table.to_csv(os.path.join(PLOT_DIR, "position_multistage_table.csv"), index=False)
order_table.to_csv(os.path.join(PLOT_DIR, "attention_order_table.csv"), index=False)
kernel_table.to_csv(os.path.join(PLOT_DIR, "spatial_kernel_table.csv"), index=False)
additional_table.to_csv(os.path.join(PLOT_DIR, "additional_attention_table.csv"), index=False)

# Export a combined training/validation history table for every model trained in the notebook
history_rows = []
for model_name, history in histories.items():
    for epoch_idx in range(len(history.history.get("loss", []))):
        row = {"model": model_name, "epoch": epoch_idx + 1}
        for metric_name, values in history.history.items(): row[metric_name] = values[epoch_idx]
        history_rows.append(row)
history_df = pd.DataFrame(history_rows)
history_df.to_csv(os.path.join(PLOT_DIR, "training_validation_histories.csv"), index=False)


# Package only the figures needed for the lab-record visualizations.
# The experiment code and generated outputs above are left unchanged.
import zipfile
from pathlib import Path

archive_base = os.path.join(str(OUTPUT_ROOT), "CS3807_Lab_Record_Required_Images")
archive_path = archive_base + ".zip"

# Figure mapping from the notebook's existing save order:
# 001: sample inputs from all five classes
# 003-004: baseline training/validation accuracy and loss
# 009-010: Channel+Spatial P3 training/validation accuracy and loss
# 035-038: graphical performance comparisons
# 039-040: baseline and selected-attention confusion matrices
# 041-045: P2/P3/P4 maps, spatial map/overlay, channel weights,
#         self-attention response, and before/after feature maps
# 046: baseline vs attention-model Grad-CAM
# 047-056: qualitative comparison on three correctly and two incorrectly
#          classified test images (Grad-CAM comparisons and labeled inputs)
required_figure_numbers = [
    1, 3, 4, 9, 10,
    35, 36, 37, 38,
    39, 40,
    41, 42, 43, 44, 45,
    46,
    47, 48, 49, 50, 51, 52, 53, 54, 55, 56,
]

plot_dir = Path(PLOT_DIR)
if os.path.exists(archive_path):
    os.remove(archive_path)

missing_figures = []
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for number in required_figure_numbers:
        figure_path = plot_dir / f"figure_{number:03d}.eps"
        if figure_path.is_file():
            zf.write(figure_path, arcname=figure_path.name)
        else:
            missing_figures.append(figure_path.name)

print(f"Required figures included: {len(required_figure_numbers) - len(missing_figures)}")
print(f"ZIP contains images only; model checkpoints, CSVs, and extra figures are excluded.")
print("Archive:", archive_path)
if missing_figures:
    print("Figures not found (run the corresponding notebook cells first):", missing_figures)
